# EDA: Kaggle Spotify Tracks

**Källa:** Kaggle, maharshipandya/spotify-tracks-dataset (`data/raw/dataset.csv`, committas inte)
**Syfte:** bedöma om datasetet duger som bas för quizet.

## Sammanfattning
- 114 000 rader, 21 kolumner, 89 741 unika `track_id`
- Nästan inga saknade värden
- Innehåller artist, låt, album, genre (114 st) och popularity
- Saknar år/datum, så det måste hämtas från annan källa (t.ex. MusicBrainz)
- 24 259 dubbletter på `track_id`: samma låt ligger en gång per genre
- Popularity ≥ 50: 29 367 rader, 21 210 unika låtar
- `artists` är en sträng med `;` mellan namnen och måste delas upp

## Slutsats
Bra bas för artist, låt, album och genre. Behöver avdubbleras och kompletteras med år.

## Öppna frågor
- Dubbletter på låtnamn + artist (olika `track_id`)?
- Vilken popularity-gräns ska vi använda?

In [6]:
import pandas as pd
import os
print(os.getcwd())
print(os.listdir('../data/raw'))

df = pd.read_csv('../data/raw/dataset.csv')

/Users/mirandalyng/Desktop/Data_engineer/GitHub_repos/examensarbete/eda
['dataset.csv']


In [7]:
df.shape

(114000, 21)

In [8]:
df.columns.tolist()

['Unnamed: 0',
 'track_id',
 'artists',
 'album_name',
 'track_name',
 'popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'track_genre']

In [9]:
df["track_id"].nunique()

89741

In [10]:
df.isna().sum()

Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [11]:
df["popularity"].describe()

count    114000.000000
mean         33.238535
std          22.305078
min           0.000000
25%          17.000000
50%          35.000000
75%          50.000000
max         100.000000
Name: popularity, dtype: float64

In [12]:
df["track_genre"].nunique()

114

In [13]:
[c for c in df.columns if "year" in c.lower() or "date" in c.lower()]

[]

In [14]:
df.duplicated("track_id").sum()
(df["popularity"] >= 50).sum()
df[df["popularity"] >= 50]["track_id"].nunique()
df["artists"].head(10)

0                             Gen Hoshino
1                            Ben Woodward
2                  Ingrid Michaelson;ZAYN
3                            Kina Grannis
4                        Chord Overstreet
5                            Tyrone Wells
6    A Great Big World;Christina Aguilera
7                              Jason Mraz
8               Jason Mraz;Colbie Caillat
9                          Ross Copperman
Name: artists, dtype: str

In [15]:
print(df.duplicated("track_id").sum())
print((df["popularity"] >= 50).sum())
print(df[df["popularity"] >= 50]["track_id"].nunique())

24259
29367
21210


## Spotify Track ID 

### Testa att skapa Spotify-länkar utan API

In [17]:
display(
    df[["track_id", "track_name", "artists", "track_genre"]].head(10)
)

,track_id,track_name,artists,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,acoustic
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,acoustic
5,01MVOl9KtVTNfFiBU9I7dc,Days I Will Remember,Tyrone Wells,acoustic
6,6Vc5wAMmXdKIAM7WUoEb7N,Say Something,A Great Big World;Christina Aguilera,acoustic
7,1EzrEOXmMH3G43AXT1y7pA,I'm Yours,Jason Mraz,acoustic
8,0IktbUcnAGrvD03AWnz3Q8,Lucky,Jason Mraz;Colbie Caillat,acoustic
9,7k9GuJYLp2AzqokyEdwEw2,Hunger,Ross Copperman,acoustic


In [25]:
df["spotify_url"] = (
    "https://open.spotify.com/track/" + df["track_id"].astype(str)
)

display(
    df[["track_name", "artists", "track_id", "spotify_url"]].head(10)
)

,track_name,artists,track_id,spotify_url
0,Comedy,Gen Hoshino,5SuOikwiRyPMVoIQDJUgSV,https://open.spotify.com/track/5SuOikwiRyPMVoI...
1,Ghost - Acoustic,Ben Woodward,4qPNDBW1i3p13qLCt0Ki3A,https://open.spotify.com/track/4qPNDBW1i3p13qL...
2,To Begin Again,Ingrid Michaelson;ZAYN,1iJBSr7s7jYXzM8EGcbK5b,https://open.spotify.com/track/1iJBSr7s7jYXzM8...
3,Can't Help Falling In Love,Kina Grannis,6lfxq3CG4xtTiEg7opyCyx,https://open.spotify.com/track/6lfxq3CG4xtTiEg...
4,Hold On,Chord Overstreet,5vjLSffimiIP26QG5WcN2K,https://open.spotify.com/track/5vjLSffimiIP26Q...
5,Days I Will Remember,Tyrone Wells,01MVOl9KtVTNfFiBU9I7dc,https://open.spotify.com/track/01MVOl9KtVTNfFi...
6,Say Something,A Great Big World;Christina Aguilera,6Vc5wAMmXdKIAM7WUoEb7N,https://open.spotify.com/track/6Vc5wAMmXdKIAM7...
7,I'm Yours,Jason Mraz,1EzrEOXmMH3G43AXT1y7pA,https://open.spotify.com/track/1EzrEOXmMH3G43A...
8,Lucky,Jason Mraz;Colbie Caillat,0IktbUcnAGrvD03AWnz3Q8,https://open.spotify.com/track/0IktbUcnAGrvD03...
9,Hunger,Ross Copperman,7k9GuJYLp2AzqokyEdwEw2,https://open.spotify.com/track/7k9GuJYLp2Azqok...


In [23]:
track_id = df.iloc[0]["track_id"]

url = f"https://open.spotify.com/track/{track_id}"

print(url)

https://open.spotify.com/track/5SuOikwiRyPMVoIQDJUgSV
